## Importing the libraries and dataset


Credit: VitorKamada - for the dataset


In [3]:
!pip install panda==1.3.5  # Install specific Pandas version for data manipulation
!pip install numpy==1.21.2  # Install specific NumPy version for numerical operations
!pip install plotly==5.10.0  # Install Plotly for interactive visualizations
!pip install statsmodels==0.13.2  # Install Statsmodels for statistical regression models
!pip install linearmodels==4.27  # Install linearmodels for IV/2SLS instrumental variable regression

In [1]:
import pandas as pd  # Import Pandas for data loading and manipulation
import numpy as np  # Import NumPy for numerical and array operations
%matplotlib inline  # Render Matplotlib plots inline inside the notebook
import plotly.express as px  # Import Plotly Express for interactive scatter plots
import statsmodels.formula.api as smf  # Import Statsmodels formula API for OLS regression
# from google.colab import drive  # (Commented out) Mount Google Drive in Colab environment
# drive.mount('/content/drive')  # (Commented out) Mount Google Drive at specified path
path = "https://github.com/VitorKamada/ECO6100/raw/master/Data/"  # Set base URL path to the raw GitHub dataset directory
df = pd.read_stata(path + "AEJfigs.dta")  # Load Stata .dta dataset from the remote GitHub URL

# RDD example

In [ ]:
df = df.dropna()  # Drop all rows with missing (NaN) values
df.head()  # Display first 5 rows of the sharp RDD dataset

,agecell,all,allfitted,internal,internalfitted,external,externalfitted,alcohol,alcoholfitted,homicide,homicidefitted,suicide,suicidefitted,mva,mvafitted,drugs,drugsfitted,externalother,externalotherfitted
0,19.068493,92.825401,91.706146,16.617590,16.738131,76.207817,74.968010,0.639138,0.794344,16.316818,16.284573,11.203714,11.592100,35.829327,34.817780,3.872425,3.448835,8.534373,8.388236
1,19.150684,95.100739,91.883720,18.327684,16.920654,76.773056,74.963066,0.677409,0.837575,16.859964,16.270697,12.193368,11.593611,35.639256,34.633888,3.236511,3.470022,8.655786,8.530174
2,19.232876,92.144295,92.049065,18.911053,17.098843,73.233238,74.950226,0.866443,0.877835,15.219254,16.262882,11.715812,11.595129,34.205650,34.446735,3.202071,3.492069,8.513741,8.662681
3,19.315069,88.427757,92.202141,16.101770,17.272680,72.325981,74.929466,0.867308,0.915115,16.742825,16.261148,11.275010,11.596655,32.278957,34.256302,3.280689,3.514980,8.258285,8.785728
4,19.397261,88.704941,92.342918,17.363520,17.442156,71.341415,74.900757,1.019163,0.949407,14.947726,16.265511,10.984314,11.598189,32.650967,34.062588,3.548198,3.538755,8.417533,8.899288


In [ ]:
fig = px.scatter(x = df.agecell, y = df['all'])  # Create scatter plot of age (running variable) vs all-cause mortality rate
fig.show()  # Render and display the interactive Plotly scatter plot

In [ ]:
df['Group'] = np.where(df['agecell']>=21,"Treatment","Control")  # Create binary Group column: 'Treatment' if age>=21 else 'Control' (cutoff=21)
df['Age'] = df['agecell']-21  # Center the running variable (age) around the cutoff of 21

In [ ]:
import statsmodels.formula.api as smf  # Import Statsmodels formula API for OLS regression

rd = "all~1 + C(Group) + Age"  # Define simple RDD formula: outcome ~ intercept + treatment group + centered age
reg_disc = smf.ols(rd,df).fit(cov_type = 'HC1')  # Fit linear OLS RDD model with HC1 heteroskedasticity-robust standard errors
print(reg_disc.summary())  # Print full OLS regression results including coefficients and p-values

                            OLS Regression Results                            
Dep. Variable:                    all   R-squared:                       0.595
Model:                            OLS   Adj. R-squared:                  0.577
Method:                 Least Squares   F-statistic:                     32.55
Date:                Mon, 22 Aug 2022   Prob (F-statistic):           1.81e-09
Time:                        05:36:43   Log-Likelihood:                -110.41
No. Observations:                  48   AIC:                             226.8
Df Residuals:                      45   BIC:                             232.4
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                91.84

In [ ]:
### Adding the quadratic terms allow for more flexibility on how the variables can interact - in OLS we force a line


In [ ]:
rd = "all~1 + Age*C(Group) + I(Age**2)*C(Group)"  # Define quadratic RDD formula: adds age^2 terms interacted with Group for flexible fit
#rd = "all~1 + Age*C(Group) + Age + C(Group) + I(Age**2)*C(Group) + Age**2"  # (Commented out) Alternative explicit quadratic formula expansion
reg_disc = smf.ols(rd,df).fit(cov_type='HC1')  # Fit linear OLS RDD model with HC1 heteroskedasticity-robust standard errors
print(reg_disc.summary())  # Print full OLS regression results including coefficients and p-values


                            OLS Regression Results                            
Dep. Variable:                    all   R-squared:                       0.682
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     19.90
Date:                Mon, 22 Aug 2022   Prob (F-statistic):           4.02e-10
Time:                        05:42:57   Log-Likelihood:                -104.57
No. Observations:                  48   AIC:                             221.1
Df Residuals:                      42   BIC:                             232.4
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [ ]:
#Another thing to notice is that when we estimate the impact closer to the cut-off, the standard error increases but the estimates are intuitively more precise

In [ ]:
df1 = df[df['agecell']>=20]  # Filter dataset to ages >= 20 (lower bandwidth bound around cutoff 21)
df1 = df[df['agecell']<=22]  # Further restrict dataset to ages <= 22 (bandwidth window: age 20 to 22)

rd3 = "all ~ 1 + Age*C(Group) + I(Age**2)*C(Group)"  # Define quadratic RDD formula for the bandwidth-restricted sample
all2 = smf.ols(rd3,df1).fit(cov_type="HC1")  # Fit restricted-bandwidth quadratic RDD model with robust standard errors
print(all2.summary())  # Print regression results for the restricted bandwidth RDD model

                            OLS Regression Results                            
Dep. Variable:                    all   R-squared:                       0.696
Model:                            OLS   Adj. R-squared:                  0.646
Method:                 Least Squares   F-statistic:                     14.69
Date:                Mon, 22 Aug 2022   Prob (F-statistic):           2.65e-07
Time:                        05:50:29   Log-Likelihood:                -81.007
No. Observations:                  36   AIC:                             174.0
Df Residuals:                      30   BIC:                             183.5
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

**Important things:**

- Parametric estimation x Non-Parametric: In the previous course, we used bayesian networks, that are non-parametric methods. The difference is that in parametric methods, we have to have some form of information about how the relationship between variables (the function) is. That is why we play around more adding the quadratic terms;

- RDD Checks - do other variables jump at the cut-off


# Fuzzy RDD Example

Dataset and example source: https://evalf20.classes.andrewheiss.com/example/rdd-fuzzy/

IV2SLS documentation: https://bashtage.github.io/linearmodels/iv/examples/using-formulas.html

In [ ]:
df_fuzzy=pd.read_csv('tutoring_program_fuzzy.csv')  # Load fuzzy RDD tutoring program CSV dataset

In [ ]:
df_fuzzy.head(2)  # Display first 2 rows of the fuzzy RDD dataset

,id,entrance_exam,tutoring,tutoring_text,exit_exam
0,1,92.408325,False,No tutor,78.075917
1,2,72.772380,False,No tutor,58.217569


In [ ]:
fig = px.scatter(x = df_fuzzy.entrance_exam, y = df_fuzzy['tutoring'], color = df_fuzzy.tutoring)  # Create scatter plot of entrance exam score vs tutoring participation
fig.show()  # Render and display the interactive Plotly scatter plot

In [ ]:
df_fuzzy['entrance_centered'] = df_fuzzy['entrance_exam'] - 70  # Center entrance exam score around cutoff of 70 (running variable)
df_fuzzy['V'] = np.where(df_fuzzy.entrance_exam<70,1,0)  # Create binary instrument V=1 if score<70 (eligible for tutoring), else 0

df_fuzzy.head(2)  # Display first 2 rows of the fuzzy RDD dataset

,id,entrance_exam,tutoring,tutoring_text,exit_exam,entrance_centered,V
0,1,92.408325,False,No tutor,78.075917,22.408325,0
1,2,72.772380,False,No tutor,58.217569,2.772380,0


In [ ]:
rd = "exit_exam~1 +C(tutoring) + entrance_centered"  # Define naive OLS formula: exit exam ~ intercept + tutoring + centered score
reg_disc = smf.ols(rd,df_fuzzy).fit(cov_type='HC1')  # Fit naive OLS ignoring non-compliance (biased due to fuzzy assignment)
print(reg_disc.summary())  # Print full OLS regression results including coefficients and p-values

                            OLS Regression Results                            
Dep. Variable:              exit_exam   R-squared:                       0.423
Model:                            OLS   Adj. R-squared:                  0.422
Method:                 Least Squares   F-statistic:                     406.3
Date:                Mon, 22 Aug 2022   Prob (F-statistic):          8.80e-130
Time:                        06:24:06   Log-Likelihood:                -3290.9
No. Observations:                1000   AIC:                             6588.
Df Residuals:                     997   BIC:                             6603.
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              59.1079    

In [ ]:
#!pip install linearmodels  # (Optional) Install linearmodels if not already installed
from linearmodels.iv import IV2SLS  # Import IV2SLS for Two-Stage Least Squares instrumental variable estimation

formula = (  # Begin defining the IV2SLS formula for fuzzy RDD
    "exit_exam ~ 1 + entrance_centered + [tutoring ~ V]"  # Center running variable around cutoff
)

mod = IV2SLS.from_formula(formula, df_fuzzy)  # Build IV2SLS model using eligibility (V) as instrument for tutoring
iv_res = mod.fit(cov_type="robust")  # Fit 2SLS model with robust standard errors to get LATE (local ATE at cutoff)
print(iv_res)  # Print IV2SLS results: LATE estimate of tutoring effect on exit exam score

                          IV-2SLS Estimation Summary                          
Dep. Variable:              exit_exam   R-squared:                      0.4229
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4217
No. Observations:                1000   F-statistic:                    390.52
Date:                Mon, Aug 22 2022   P-value (F-stat)                0.0000
Time:                        06:26:51   Distribution:                  chi2(2)
Cov. Estimator:                robust                                         
                                                                              
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------
Intercept             58.879     0.7814     75.353     0.0000      57.347      60.410
entrance_centered     0.